# Реранкер

Второй уровень двухступенчатой архитектуры:

- first level: ALS + top popularity
- second level: CatBoostRanker
- train: temporal split
- save: все артефакты для повторного использования

In [1]:
import gc
import os
import sys
import json
from datetime import date
from pathlib import Path

import dask.dataframe as dd
import numpy as np
import pandas as pd

from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

sys.path.append("../src")

from data_loader import (
    download_dataset,
    load_orders,
    load_interactions,
    load_items,
    load_categories,
)

## Инициализация

In [2]:
download_dataset()

Path("../models").mkdir(parents=True, exist_ok=True)
Path("../data/interim").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

Датасет загружен ранее


In [3]:
orders_df = load_orders(train=True)
tracker_df = load_interactions(train=True)
orders_ddf_test = load_orders(train=False)

items_df = load_items()
categories_df = load_categories()

test_users = orders_ddf_test["user_id"].unique().compute().sort_values().to_list()

print(f"test_users: {len(test_users):,}")

test_users: 346,759


## Конфиг

In [4]:
# ------------------------------------------------------------
# SPLITS
# ------------------------------------------------------------
TRAIN_START = date(2025, 4, 18)
TRAIN_END   = date(2025, 6, 17)

VAL_START   = date(2025, 6, 18)
VAL_END     = date(2025, 7, 1)

NEW_TRAIN_START = date(2025, 5, 2)
NEW_TRAIN_END   = date(2025, 7, 1)

# ------------------------------------------------------------
# CANDIDATES / RERANK
# ------------------------------------------------------------
FINAL_TOPK = 100

RERANK_SEED = 42
CB_ITER = 500
CB_DEPTH = 6
CB_LR = 0.05
CB_OD_WAIT = 50

EVENT_WEIGHTS = {
    "page_view": 1.0,
    "view_description": 2.0,
    "review_view": 3.0,
    "favorite": 20.0,
    "to_cart": 100.0,
    "order": 100.0,
}

DECAY_TAU_DAYS = 14.0
DECAY_MIN_FACTOR = 0.10

np.random.seed(RERANK_SEED)

## Общие helper-функции

In [5]:
def to_pandas_if_dask(df):
    if isinstance(df, dd.DataFrame):
        return df.compute()
    return df.copy()


def filter_tracker_ddf(
    tracker_ddf: dd.DataFrame,
    start_date: date,
    end_date: date,
    keep_date: bool = False,
) -> dd.DataFrame:
    tracker_ddf = tracker_ddf.assign(
        date=dd.to_datetime(tracker_ddf["date"], format="%Y-%m-%d")
    )

    filtered = tracker_ddf[
        (tracker_ddf["date"] >= pd.to_datetime(start_date))
        & (tracker_ddf["date"] <= pd.to_datetime(end_date))
    ]

    cols = ["user_id", "item_id", "action_type"]
    if keep_date:
        cols.append("date")
    return filtered[cols]


def filter_orders_ddf(
    orders_ddf: dd.DataFrame,
    start_date: date,
    end_date: date,
    keep_created_date: bool = False,
) -> dd.DataFrame:
    orders_ddf = orders_ddf.assign(
        created_date=dd.to_datetime(orders_ddf["created_date"], format="%Y-%m-%d")
    )

    filtered = orders_ddf[
        (orders_ddf["created_date"] >= pd.to_datetime(start_date))
        & (orders_ddf["created_date"] <= pd.to_datetime(end_date))
    ]

    cols = ["user_id", "item_id"]
    if keep_created_date:
        cols.append("created_date")
    return filtered[cols]


def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance)[:k]
    if relevance.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, relevance.size + 2))
    return float(np.sum(relevance * discounts))


def ndcg_at_k(recommended_items, relevant_items, k=100):
    if not relevant_items:
        return None

    relevance = [1 if item in relevant_items else 0 for item in recommended_items[:k]]
    dcg = dcg_at_k(relevance, k)

    ideal_relevance = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal_relevance, k)

    return dcg / idcg if idcg > 0 else 0.0


def decay_from_age(age_days: np.ndarray,
                   tau_days: float = DECAY_TAU_DAYS,
                   min_factor: float = DECAY_MIN_FACTOR) -> np.ndarray:
    x = np.exp(-age_days / tau_days)
    return np.clip(x, min_factor, 1.0).astype(np.float32)


def inv_rank(x: pd.Series) -> pd.Series:
    x = x.replace(0, np.nan)
    return (1.0 / x).fillna(0.0).astype(np.float32)


def maybe_build_test_gt(orders_ddf_test):
    if "item_id" not in orders_ddf_test.columns:
        return None

    sample = orders_ddf_test["item_id"].dropna()
    try:
        sample_n = sample.shape[0].compute()
    except Exception:
        return None

    if sample_n == 0:
        return None

    user2items_test = (
        orders_ddf_test.groupby("user_id")["item_id"]
        .apply(list, meta=("item_id", "object"))
        .compute()
        .to_dict()
    )
    return {int(k): list(map(int, v)) for k, v in user2items_test.items()}

## Meta и артефакты first level

In [6]:
def extract_cat_l0(ids):
    if ids is None or (isinstance(ids, float) and np.isnan(ids)):
        return np.nan
    if isinstance(ids, (list, tuple, np.ndarray)):
        clean = [x for x in ids if x != -1]
        return clean[0] if len(clean) else np.nan
    return np.nan


def build_item_meta(items_df, categories_df):
    items_pdf = to_pandas_if_dask(items_df)
    cat_pdf = to_pandas_if_dask(categories_df)

    cat_pdf = cat_pdf.copy()
    cat_pdf["cat_l0"] = cat_pdf["ids"].apply(extract_cat_l0)

    item_meta = (
        items_pdf[["item_id", "catalogid"]]
        .drop_duplicates("item_id")
        .merge(
            cat_pdf[["catalogid", "cat_l0"]].drop_duplicates("catalogid"),
            on="catalogid",
            how="left",
        )
        [["item_id", "cat_l0"]]
        .drop_duplicates("item_id")
    )
    return item_meta


def load_als_artifacts(kind: str):
    if kind == "val":
        candidates_path = "../data/interim/als_val_candidates.parquet"
        model_path = "../models/als_model_for_val.npz"
        user_map_path = "../models/als_val_user_mapping.parquet"
        item_map_path = "../models/als_val_item_mapping.parquet"
    elif kind == "full":
        candidates_path = "../data/interim/als_test_candidates.parquet"
        model_path = "../models/als_model.npz"
        user_map_path = "../models/als_user_mapping.parquet"
        item_map_path = "../models/als_item_mapping.parquet"
    else:
        raise ValueError("kind must be 'val' or 'full'")

    candidates = pd.read_parquet(candidates_path).copy()
    user_map = pd.read_parquet(user_map_path)
    item_map = pd.read_parquet(item_map_path)

    required_cols = {"user_id", "item_id", "als_score", "als_rank", "in_als"}
    missing = required_cols - set(candidates.columns)
    if missing:
        raise ValueError(
            f"В {candidates_path} не хватает колонок: {sorted(missing)}"
        )

    bundle = {
        "candidates": candidates,
        "model_path": model_path,
        "user_map": user_map,
        "item_map": item_map,
    }
    return bundle


def load_pop_artifacts(kind: str):
    if kind == "val":
        pop_table_path = "../models/popular_items_val.parquet"
        pop_recs_path = "../data/interim/pop_val_recommendations.parquet"
    elif kind == "full":
        pop_table_path = "../models/popular_items.parquet"
        pop_recs_path = "../data/interim/pop_test_recommendations.parquet"
    else:
        raise ValueError("kind must be 'val' or 'full'")

    pop_table = pd.read_parquet(pop_table_path).copy()
    pop_recs = pd.read_parquet(pop_recs_path).copy()

    if "rank" in pop_recs.columns:
        pop_recs = pop_recs.rename(columns={"rank": "pop_rank"})

    if "score" in pop_table.columns:
        pop_table = pop_table.rename(columns={"score": "pop_score"})

    required_pop_table = {"item_id", "pop_score", "global_pop_rank"}
    missing_table = required_pop_table - set(pop_table.columns)
    if missing_table:
        raise ValueError(
            f"В {pop_table_path} не хватает колонок: {sorted(missing_table)}"
        )

    required_pop_recs = {"user_id", "item_id", "pop_rank"}
    missing_recs = required_pop_recs - set(pop_recs.columns)
    if missing_recs:
        raise ValueError(
            f"В {pop_recs_path} не хватает колонок: {sorted(missing_recs)}"
        )

    pop_recs["in_pop"] = 1

    pop_recs = pop_recs.merge(
        pop_table[["item_id", "pop_score", "global_pop_rank"]].drop_duplicates("item_id"),
        on="item_id",
        how="left",
    )

    return pop_table, pop_recs


def merge_candidate_sources(als_df: pd.DataFrame, pop_df: pd.DataFrame) -> pd.DataFrame:
    merged = als_df.merge(
        pop_df,
        on=["user_id", "item_id"],
        how="outer",
    )

    defaults = {
        "als_score": 0.0,
        "als_rank": 0,
        "in_als": 0,
        "pop_score": 0.0,
        "pop_rank": 0,
        "in_pop": 0,
        "global_pop_rank": 0,
    }
    for c, v in defaults.items():
        merged[c] = merged[c].fillna(v)

    merged["both_sources"] = ((merged["in_als"] + merged["in_pop"]) == 2).astype(np.int8)
    return merged


def build_labels_df(future_orders_pdf: pd.DataFrame) -> pd.DataFrame:
    return (
        future_orders_pdf[["user_id", "item_id"]]
        .drop_duplicates()
        .assign(label=1.0)
    )


def candidate_recall(candidates_df: pd.DataFrame, labels_df: pd.DataFrame) -> float:
    cand_sets = candidates_df.groupby("user_id")["item_id"].apply(set).to_dict()
    gt_sets = labels_df.groupby("user_id")["item_id"].apply(set).to_dict()

    recalls = []
    for u, gt in gt_sets.items():
        if len(gt) == 0:
            continue
        cand = cand_sets.get(u, set())
        recalls.append(len(cand & gt) / len(gt))

    return float(np.mean(recalls)) if recalls else np.nan

## Построение feature tables

In [7]:
def prepare_events(
    tracker_pdf: pd.DataFrame,
    orders_pdf: pd.DataFrame,
    ref_date: pd.Timestamp,
) -> pd.DataFrame:
    tracker = tracker_pdf.copy()
    orders = orders_pdf.copy()

    tracker["event_ts"] = pd.to_datetime(tracker["date"])
    orders["event_ts"] = pd.to_datetime(orders["created_date"])

    tracker["base_weight"] = (
        tracker["action_type"].map(EVENT_WEIGHTS).fillna(1.0).astype(np.float32)
    )
    tracker_age = (ref_date - tracker["event_ts"]).dt.days.clip(lower=0).astype(np.float32).values
    tracker["strength"] = tracker["base_weight"].values * decay_from_age(tracker_age)

    orders["action_type"] = "order"
    orders["base_weight"] = np.float32(EVENT_WEIGHTS["order"])
    orders_age = (ref_date - orders["event_ts"]).dt.days.clip(lower=0).astype(np.float32).values
    orders["strength"] = orders["base_weight"].values * decay_from_age(orders_age)

    events = pd.concat(
        [
            tracker[["user_id", "item_id", "action_type", "event_ts", "strength"]],
            orders[["user_id", "item_id", "action_type", "event_ts", "strength"]],
        ],
        ignore_index=True,
    )
    events["age_days"] = (ref_date - events["event_ts"]).dt.days.clip(lower=0).astype(np.int16)
    return events


def build_feature_tables(
    tracker_pdf: pd.DataFrame,
    orders_pdf: pd.DataFrame,
    item_meta: pd.DataFrame,
    ref_date: pd.Timestamp,
    cand_df: pd.DataFrame | None = None,
):
    if cand_df is None:
        raise ValueError("Для build_feature_tables нужен cand_df")

    tracker_pdf = tracker_pdf[["user_id", "item_id", "action_type", "date"]].copy()
    orders_pdf = orders_pdf[["user_id", "item_id", "created_date"]].copy()
    item_meta_small = item_meta[["item_id", "cat_l0"]].drop_duplicates().copy()

    cand_pairs = cand_df[["user_id", "item_id"]].drop_duplicates().copy()
    cand_users = cand_pairs["user_id"].drop_duplicates()
    cand_items = cand_pairs["item_id"].drop_duplicates()

    # user-level history: режем только по users
    tracker_u = tracker_pdf[tracker_pdf["user_id"].isin(cand_users)].copy()
    orders_u = orders_pdf[orders_pdf["user_id"].isin(cand_users)].copy()

    # item-level history: режем только по items
    tracker_i = tracker_pdf[tracker_pdf["item_id"].isin(cand_items)].copy()
    orders_i = orders_pdf[orders_pdf["item_id"].isin(cand_items)].copy()

    # ui-level history: сначала user history, потом точный inner join по candidate pairs
    events_u = prepare_events(tracker_u, orders_u, ref_date)
    events_i = prepare_events(tracker_i, orders_i, ref_date)

    ui_events = events_u.merge(cand_pairs, on=["user_id", "item_id"], how="inner")
    ui_feat = (
        ui_events.groupby(["user_id", "item_id"], as_index=False)
        .agg(
            ui_strength=("strength", "sum"),
            ui_event_cnt=("strength", "size"),
            ui_last_ts=("event_ts", "max"),
        )
    )

    user_feat = (
        events_u.groupby("user_id", as_index=False)
        .agg(
            user_total_strength=("strength", "sum"),
            user_event_cnt=("strength", "size"),
            user_unique_items=("item_id", "nunique"),
            user_last_ts=("event_ts", "max"),
        )
    )

    item_feat = (
        events_i.groupby("item_id", as_index=False)
        .agg(
            item_total_strength=("strength", "sum"),
            item_event_cnt=("strength", "size"),
            item_unique_users=("user_id", "nunique"),
            item_last_ts=("event_ts", "max"),
        )
    )

    item_orders = orders_i.groupby("item_id", as_index=False).agg(
        item_order_cnt=("item_id", "size")
    )
    item_feat = item_feat.merge(item_orders, on="item_id", how="left")

    ui_cat = ui_events.merge(item_meta_small, on="item_id", how="left")
    user_cat_feat = (
        ui_cat.dropna(subset=["cat_l0"])
        .groupby(["user_id", "cat_l0"], as_index=False)
        .agg(user_cat_strength=("strength", "sum"))
    )
    user_cat_feat["user_cat_rank"] = (
        user_cat_feat.groupby("user_id")["user_cat_strength"]
        .rank(method="dense", ascending=False)
        .astype(np.int16)
    )

    del tracker_u, orders_u, tracker_i, orders_i, events_u, events_i, ui_events, ui_cat, item_orders
    gc.collect()

    return {
        "ui_feat": ui_feat,
        "user_feat": user_feat,
        "item_feat": item_feat,
        "user_cat_feat": user_cat_feat,
    }

In [8]:
def reduce_train_basis(train_basis: pd.DataFrame, max_neg_per_user: int = 20) -> pd.DataFrame:
    pos = train_basis[train_basis["label"] > 0].copy()
    neg = train_basis[train_basis["label"] == 0].copy()

    sort_cols = [
        "user_id", "both_sources", "in_als", "in_pop",
        "als_rank", "pop_rank", "global_pop_rank",
    ]
    asc = [True, False, False, False, True, True, True]

    neg = neg.sort_values(sort_cols, ascending=asc)
    neg = neg.groupby("user_id", group_keys=False).head(max_neg_per_user)

    out = pd.concat([pos, neg], ignore_index=True)
    out = out.sort_values(["user_id", "label"], ascending=[True, False]).reset_index(drop=True)
    return out

In [9]:
def make_rerank_features(
    cand_df: pd.DataFrame,
    feature_tables: dict,
    item_meta: pd.DataFrame,
    pop_table: pd.DataFrame,
    ref_date: pd.Timestamp,
):
    df = cand_df.copy()

    ui_feat = feature_tables["ui_feat"].copy()
    user_feat = feature_tables["user_feat"].copy()
    item_feat = feature_tables["item_feat"].copy()
    user_cat_feat = feature_tables["user_cat_feat"].copy()

    # global_pop_rank ждём из candidate basis
    # из pop_table подтягиваем только score
    pop_item = pop_table[["item_id", "pop_score"]].copy()
    pop_item = pop_item.rename(columns={"pop_score": "global_pop_score"})

    df = df.merge(item_meta, on="item_id", how="left")
    df = df.merge(ui_feat, on=["user_id", "item_id"], how="left")
    df = df.merge(user_feat, on="user_id", how="left")
    df = df.merge(item_feat, on="item_id", how="left")
    df = df.merge(pop_item, on="item_id", how="left")
    df = df.merge(user_cat_feat, on=["user_id", "cat_l0"], how="left")

    df["ui_days_since_last"] = (ref_date - pd.to_datetime(df["ui_last_ts"])).dt.days
    df["user_days_since_last"] = (ref_date - pd.to_datetime(df["user_last_ts"])).dt.days
    df["item_days_since_last"] = (ref_date - pd.to_datetime(df["item_last_ts"])).dt.days

    required_candidate_cols = [
        "als_score", "als_rank", "in_als",
        "pop_score", "pop_rank", "in_pop",
        "global_pop_rank",
        "both_sources",
    ]
    missing_required = [c for c in required_candidate_cols if c not in df.columns]
    if missing_required:
        raise ValueError(f"В candidate basis не хватает колонок: {missing_required}")

    zero_cols = [
        "ui_strength", "ui_event_cnt",
        "user_total_strength", "user_event_cnt", "user_unique_items",
        "item_total_strength", "item_event_cnt", "item_unique_users", "item_order_cnt",
        "user_cat_strength", "user_cat_rank",
    ]
    for c in zero_cols:
        if c not in df.columns:
            df[c] = 0
        df[c] = df[c].fillna(0)

    for c in [
        "als_score", "als_rank", "in_als",
        "pop_score", "pop_rank", "in_pop",
        "global_pop_score", "global_pop_rank",
        "both_sources",
    ]:
        if c not in df.columns:
            df[c] = 0
        df[c] = df[c].fillna(0)

    for c in ["ui_days_since_last", "user_days_since_last", "item_days_since_last"]:
        if c not in df.columns:
            df[c] = 9999
        df[c] = df[c].fillna(9999)

    df["als_rank_inv"] = inv_rank(df["als_rank"])
    df["pop_rank_inv"] = inv_rank(df["pop_rank"])
    df["global_pop_rank_inv"] = inv_rank(df["global_pop_rank"])
    df["source_count"] = (df["in_als"] + df["in_pop"]).astype(np.float32)
    df["same_cat_top3"] = df["user_cat_rank"].between(1, 3).astype(np.int8)
    df["user_is_cold"] = (df["user_event_cnt"] == 0).astype(np.int8)

    df["ui_strength_user_share"] = df["ui_strength"] / (df["user_total_strength"] + 1e-6)
    df["user_cat_share"] = df["user_cat_strength"] / (df["user_total_strength"] + 1e-6)
    df["item_order_user_event_ratio"] = df["item_order_cnt"] / (df["user_event_cnt"] + 1e-6)

    log_cols = [
        "als_score", "pop_score", "global_pop_score",
        "ui_strength", "user_total_strength", "item_total_strength",
        "ui_event_cnt", "user_event_cnt", "item_event_cnt",
        "item_order_cnt", "item_unique_users", "user_unique_items",
        "user_cat_strength",
    ]
    for c in log_cols:
        df[f"log1p_{c}"] = np.log1p(df[c].astype(np.float32).clip(lower=0))

    feature_cols = [
        "in_als", "in_pop", "both_sources", "source_count",
        "als_score", "als_rank_inv",
        "pop_score", "pop_rank_inv",
        "global_pop_score", "global_pop_rank_inv",

        "ui_strength", "ui_event_cnt", "ui_strength_user_share", "ui_days_since_last",
        "user_total_strength", "user_event_cnt", "user_unique_items", "user_days_since_last", "user_is_cold",
        "item_total_strength", "item_event_cnt", "item_unique_users", "item_order_cnt", "item_days_since_last",
        "item_order_user_event_ratio",
        "user_cat_strength", "user_cat_share", "same_cat_top3",

        "log1p_als_score", "log1p_pop_score", "log1p_global_pop_score",
        "log1p_ui_strength", "log1p_user_total_strength", "log1p_item_total_strength",
        "log1p_ui_event_cnt", "log1p_user_event_cnt", "log1p_item_event_cnt",
        "log1p_item_order_cnt", "log1p_item_unique_users", "log1p_user_unique_items",
        "log1p_user_cat_strength",
    ]

    keep_cols = ["user_id", "item_id"] + feature_cols
    if "label" in df.columns:
        keep_cols = ["user_id", "item_id", "label"] + feature_cols

    out = df[keep_cols].copy()

    out["user_id"] = pd.to_numeric(out["user_id"], downcast="integer")
    out["item_id"] = pd.to_numeric(out["item_id"], downcast="integer")
    if "label" in out.columns:
        out["label"] = out["label"].astype(np.float32)

    for c in feature_cols:
        out[c] = out[c].astype(np.float32)

    return out, feature_cols

## Обучение и оценка реранкера

In [10]:
def rerank_candidates(
    df: pd.DataFrame,
    feature_cols: list[str],
    ranker: CatBoostRanker,
    topk: int = FINAL_TOPK,
) -> pd.DataFrame:
    scored = df.copy()
    scored["rerank_score"] = ranker.predict(scored[feature_cols])

    scored = (
        scored.sort_values(["user_id", "rerank_score"], ascending=[True, False])
        .groupby("user_id")
        .head(topk)
        .copy()
    )
    scored["rank"] = scored.groupby("user_id").cumcount() + 1
    return scored


def evaluate_ranked_df(scored_df: pd.DataFrame, labels_df: pd.DataFrame, k: int = FINAL_TOPK) -> float:
    pred_map = (
        scored_df.sort_values(["user_id", "rank"])
        .groupby("user_id")["item_id"]
        .apply(list)
        .to_dict()
    )
    gt_map = labels_df.groupby("user_id")["item_id"].apply(set).to_dict()

    scores = []
    for u, relevant in gt_map.items():
        score = ndcg_at_k(pred_map.get(u, []), relevant, k=k)
        if score is not None:
            scores.append(score)

    return float(np.mean(scores)) if scores else np.nan


def fit_reranker(
    train_df: pd.DataFrame,
    feature_cols: list[str],
):
    train_df = train_df.sort_values(["user_id", "item_id"]).reset_index(drop=True)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RERANK_SEED)
    tr_idx, va_idx = next(gss.split(train_df, groups=train_df["user_id"]))

    tr = train_df.iloc[tr_idx].copy().sort_values(["user_id", "item_id"]).reset_index(drop=True)
    va = train_df.iloc[va_idx].copy().sort_values(["user_id", "item_id"]).reset_index(drop=True)

    train_pool = Pool(
        data=tr[feature_cols],
        label=tr["label"],
        group_id=tr["user_id"],
    )
    eval_pool = Pool(
        data=va[feature_cols],
        label=va["label"],
        group_id=va["user_id"],
    )

    base_params = dict(
        depth=CB_DEPTH,
        learning_rate=CB_LR,
        loss_function="YetiRank",
        eval_metric="NDCG:top=100",
        random_seed=RERANK_SEED,
        verbose=100,
    )

    # 1) Находим best_iteration на holdout
    probe_ranker = CatBoostRanker(
        iterations=CB_ITER,
        od_type="Iter",
        od_wait=CB_OD_WAIT,
        **base_params,
    )

    probe_ranker.fit(train_pool, eval_set=eval_pool, use_best_model=True)

    va_scored = rerank_candidates(
        va[["user_id", "item_id"] + feature_cols],
        feature_cols,
        probe_ranker,
        topk=FINAL_TOPK,
    )
    va_labels = va[va["label"] > 0][["user_id", "item_id"]].copy()
    holdout_ndcg = evaluate_ranked_df(va_scored, va_labels, k=FINAL_TOPK)

    best_iteration = probe_ranker.get_best_iteration()
    if best_iteration is None or best_iteration < 0:
        best_iteration = CB_ITER - 1

    final_iterations = int(best_iteration) + 1

    # 2) Переобучаем финальную модель на всем train_df
    full_train_pool = Pool(
        data=train_df[feature_cols],
        label=train_df["label"],
        group_id=train_df["user_id"],
    )

    final_ranker = CatBoostRanker(
        iterations=final_iterations,
        **base_params,
    )
    final_ranker.fit(full_train_pool)

    return final_ranker, holdout_ndcg, final_iterations

## Train reranker на validation split

In [11]:
item_meta = build_item_meta(items_df, categories_df)

# future labels
orders_future_val = to_pandas_if_dask(
    filter_orders_ddf(orders_df, VAL_START, VAL_END, keep_created_date=True)
)

labels_val = build_labels_df(orders_future_val)
val_users = sorted(labels_val["user_id"].drop_duplicates().astype(int).tolist())
print(f"val_users: {len(val_users):,}")

als_val_artifacts = load_als_artifacts("val")
als_val_candidates = als_val_artifacts["candidates"].copy()
als_val_candidates = als_val_candidates[
    als_val_candidates["user_id"].isin(val_users)
].copy()

pop_table_val, pop_val_recs = load_pop_artifacts("val")

union_val_candidates = merge_candidate_sources(als_val_candidates, pop_val_recs)

cand_recall_val = candidate_recall(union_val_candidates, labels_val)

print(f"ALS val candidates rows: {len(als_val_candidates):,}")
print(f"POP val candidates rows: {len(pop_val_recs):,}")
print(f"UNION val candidates rows: {len(union_val_candidates):,}")
print(f"Candidate recall on val: {cand_recall_val:.5f}")

val_users: 359,928
ALS val candidates rows: 67,857,200
POP val candidates rows: 35,992,800
UNION val candidates rows: 98,554,754
Candidate recall on val: 0.04907


In [12]:
ref_date_val = pd.to_datetime(TRAIN_END)

train_basis_val = union_val_candidates.merge(
    labels_val,
    on=["user_id", "item_id"],
    how="left",
)
train_basis_val["label"] = train_basis_val["label"].fillna(0.0).astype(np.float32)

# режем negatives
train_basis_val = reduce_train_basis(train_basis_val, max_neg_per_user=60)

cand_users_val = train_basis_val["user_id"].drop_duplicates().astype(np.int64).tolist()
cand_items_val = train_basis_val["item_id"].drop_duplicates().astype(np.int64).tolist()

tracker_hist_val_ddf = filter_tracker_ddf(tracker_df, TRAIN_START, TRAIN_END, keep_date=True)
tracker_hist_val_ddf = tracker_hist_val_ddf[
    tracker_hist_val_ddf["user_id"].isin(cand_users_val) |
    tracker_hist_val_ddf["item_id"].isin(cand_items_val)
]
tracker_hist_val = to_pandas_if_dask(tracker_hist_val_ddf)

orders_hist_val_ddf = filter_orders_ddf(orders_df, TRAIN_START, TRAIN_END, keep_created_date=True)
orders_hist_val_ddf = orders_hist_val_ddf[
    orders_hist_val_ddf["user_id"].isin(cand_users_val) |
    orders_hist_val_ddf["item_id"].isin(cand_items_val)
]
orders_hist_val = to_pandas_if_dask(orders_hist_val_ddf)

feature_tables_val = build_feature_tables(
    tracker_pdf=tracker_hist_val,
    orders_pdf=orders_hist_val,
    item_meta=item_meta,
    ref_date=ref_date_val,
    cand_df=train_basis_val,
)

train_feat_val, feature_cols = make_rerank_features(
    cand_df=train_basis_val,
    feature_tables=feature_tables_val,
    item_meta=item_meta,
    pop_table=pop_table_val,
    ref_date=ref_date_val,
)

Path("../data/interim").mkdir(parents=True, exist_ok=True)
train_basis_val.to_parquet("../data/interim/reranker_train_basis_val_light.parquet", index=False)
train_feat_val.to_parquet("../data/interim/reranker_train_feat_val_light.parquet", index=False)
with open("../data/interim/reranker_feature_cols_light.json", "w") as f:
    json.dump(feature_cols, f, ensure_ascii=False)

print(train_basis_val.shape)
print(train_feat_val.shape)
print(f"feature cols: {len(feature_cols)}")
train_feat_val.head()

(21650820, 11)
(21650820, 44)
feature cols: 41


,user_id,item_id,label,in_als,in_pop,both_sources,source_count,als_score,als_rank_inv,pop_score,...,log1p_ui_strength,log1p_user_total_strength,log1p_item_total_strength,log1p_ui_event_cnt,log1p_user_event_cnt,log1p_item_event_cnt,log1p_item_order_cnt,log1p_item_unique_users,log1p_user_unique_items,log1p_user_cat_strength
0,1,272664684,1.0,1.0,1.0,1.0,2.0,0.282370,0.125000,33073.0,...,0.000000,7.755112,10.619971,0.000000,7.559559,10.736961,5.451038,9.916009,7.013916,0.353732
1,1,77696741,0.0,1.0,1.0,1.0,2.0,0.411392,1.000000,103136.0,...,1.480464,7.755112,12.502031,1.609438,7.559559,11.915178,7.468513,11.173388,7.013916,1.480464
2,1,44053675,0.0,1.0,1.0,1.0,2.0,0.305527,0.333333,16882.0,...,0.000000,7.755112,10.016734,0.000000,7.559559,9.916552,4.248495,9.209540,7.013916,3.333150
3,1,8952890,0.0,1.0,1.0,1.0,2.0,0.291791,0.200000,23536.0,...,0.000000,7.755112,10.265705,0.000000,7.559559,10.284182,4.615120,9.517825,7.013916,3.333150
4,1,228164505,0.0,1.0,1.0,1.0,2.0,0.271784,0.100000,44476.0,...,0.000000,7.755112,12.309551,0.000000,7.559559,11.341532,7.145196,10.490996,7.013916,0.000000


In [13]:
for c in train_feat_val.columns:
    if c in ["user_id", "item_id"]:
        train_feat_val[c] = pd.to_numeric(train_feat_val[c], downcast="integer")
    elif c == "label":
        train_feat_val[c] = train_feat_val[c].astype(np.float32)
    else:
        train_feat_val[c] = pd.to_numeric(train_feat_val[c], downcast="float")

In [14]:
SAVE_DIR = Path("../data/interim/reranker_light")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

train_basis_path = SAVE_DIR / "train_basis_val_light.parquet"
train_feat_path = SAVE_DIR / "train_feat_val_light.parquet"
feature_cols_path = SAVE_DIR / "feature_cols_light.json"

train_basis_val.to_parquet(train_basis_path, index=False)
train_feat_val.to_parquet(train_feat_path, index=False)

with open(feature_cols_path, "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False)

print("saved:")
print(train_basis_path)
print(train_feat_path)
print(feature_cols_path)
print(f"train_basis_val: {train_basis_val.shape}")
print(f"train_feat_val: {train_feat_val.shape}")
print(f"feature_cols: {len(feature_cols)}")

saved:
..\data\interim\reranker_light\train_basis_val_light.parquet
..\data\interim\reranker_light\train_feat_val_light.parquet
..\data\interim\reranker_light\feature_cols_light.json
train_basis_val: (21650820, 11)
train_feat_val: (21650820, 44)
feature_cols: 41


In [15]:
for _name in [
    "feature_tables_val",
    "tracker_hist_val",
    "orders_hist_val",
    "tracker_hist_val_ddf",
    "orders_hist_val_ddf",
]:
    if _name in globals():
        del globals()[_name]

gc.collect()

33

In [16]:
SAVE_DIR = Path("../data/interim/reranker_light")

train_basis_val = pd.read_parquet(SAVE_DIR / "train_basis_val_light.parquet")
train_feat_val = pd.read_parquet(SAVE_DIR / "train_feat_val_light.parquet")

with open(SAVE_DIR / "feature_cols_light.json", "r", encoding="utf-8") as f:
    feature_cols = json.load(f)

print(train_basis_val.shape)
print(train_feat_val.shape)
print(f"feature cols: {len(feature_cols)}")
train_feat_val.head()

(21650820, 11)
(21650820, 44)
feature cols: 41


,user_id,item_id,label,in_als,in_pop,both_sources,source_count,als_score,als_rank_inv,pop_score,...,log1p_ui_strength,log1p_user_total_strength,log1p_item_total_strength,log1p_ui_event_cnt,log1p_user_event_cnt,log1p_item_event_cnt,log1p_item_order_cnt,log1p_item_unique_users,log1p_user_unique_items,log1p_user_cat_strength
0,1,272664684,1.0,1.0,1.0,1.0,2.0,0.282370,0.125000,33073.0,...,0.000000,7.755112,10.619971,0.000000,7.559559,10.736961,5.451038,9.916009,7.013916,0.353732
1,1,77696741,0.0,1.0,1.0,1.0,2.0,0.411392,1.000000,103136.0,...,1.480464,7.755112,12.502031,1.609438,7.559559,11.915178,7.468513,11.173388,7.013916,1.480464
2,1,44053675,0.0,1.0,1.0,1.0,2.0,0.305527,0.333333,16882.0,...,0.000000,7.755112,10.016734,0.000000,7.559559,9.916552,4.248495,9.209540,7.013916,3.333150
3,1,8952890,0.0,1.0,1.0,1.0,2.0,0.291791,0.200000,23536.0,...,0.000000,7.755112,10.265705,0.000000,7.559559,10.284182,4.615120,9.517825,7.013916,3.333150
4,1,228164505,0.0,1.0,1.0,1.0,2.0,0.271784,0.100000,44476.0,...,0.000000,7.755112,12.309551,0.000000,7.559559,11.341532,7.145196,10.490996,7.013916,0.000000


In [17]:
ranker, holdout_ndcg, final_iterations = fit_reranker(
    train_df=train_feat_val[["user_id", "item_id", "label"] + feature_cols],
    feature_cols=feature_cols,
)

train_scored_val = rerank_candidates(
    df=train_feat_val[["user_id", "item_id"] + feature_cols],
    feature_cols=feature_cols,
    ranker=ranker,
    topk=FINAL_TOPK,
)

train_ndcg = evaluate_ranked_df(
    scored_df=train_scored_val,
    labels_df=labels_val,
    k=FINAL_TOPK,
)

print(f"Holdout NDCG@{FINAL_TOPK}: {holdout_ndcg:.5f}")
print(f"Train-basis NDCG@{FINAL_TOPK}: {train_ndcg:.5f}")
print(f"Final CatBoost iterations: {final_iterations}")

0:	test: 0.9676866	best: 0.9676866 (0)	total: 2.37s	remaining: 19m 42s
100:	test: 0.9751217	best: 0.9751329 (98)	total: 3m 35s	remaining: 14m 10s
200:	test: 0.9753251	best: 0.9753487 (176)	total: 7m 5s	remaining: 10m 33s
300:	test: 0.9753849	best: 0.9754179 (291)	total: 10m 37s	remaining: 7m 1s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9754344502
bestIteration = 324

Shrink model to first 325 iterations.
0:	total: 2.54s	remaining: 13m 43s
100:	total: 4m 13s	remaining: 9m 22s
200:	total: 8m 22s	remaining: 5m 10s
300:	total: 12m 32s	remaining: 60s
324:	total: 13m 31s	remaining: 0us
Holdout NDCG@100: 0.80107
Train-basis NDCG@100: 0.09953
Final CatBoost iterations: 325


In [18]:
def evaluate_reranker_on_full_val_batched(
    union_val_candidates: pd.DataFrame,
    labels_val: pd.DataFrame,
    tracker_df,
    orders_df,
    item_meta: pd.DataFrame,
    pop_table_val: pd.DataFrame,
    ranker,
    feature_cols: list[str],
    ref_date_val: pd.Timestamp,
    batch_users: int = 5000,
    topk: int = FINAL_TOPK,
):
    gt_map = labels_val.groupby("user_id")["item_id"].apply(set).to_dict()
    val_user_list = union_val_candidates["user_id"].drop_duplicates().astype(np.int64).tolist()

    ndcg_scores = []

    for start in tqdm(range(0, len(val_user_list), batch_users), desc="full val eval batches"):
        batch_user_ids = val_user_list[start:start + batch_users]

        batch_cand = union_val_candidates[
            union_val_candidates["user_id"].isin(batch_user_ids)
        ].copy()
        batch_item_ids = batch_cand["item_id"].drop_duplicates().astype(np.int64).tolist()

        tracker_hist_batch_ddf = filter_tracker_ddf(
            tracker_df, TRAIN_START, TRAIN_END, keep_date=True
        )
        tracker_hist_batch_ddf = tracker_hist_batch_ddf[
            tracker_hist_batch_ddf["user_id"].isin(batch_user_ids) |
            tracker_hist_batch_ddf["item_id"].isin(batch_item_ids)
        ]
        tracker_hist_batch = to_pandas_if_dask(tracker_hist_batch_ddf)

        orders_hist_batch_ddf = filter_orders_ddf(
            orders_df, TRAIN_START, TRAIN_END, keep_created_date=True
        )
        orders_hist_batch_ddf = orders_hist_batch_ddf[
            orders_hist_batch_ddf["user_id"].isin(batch_user_ids) |
            orders_hist_batch_ddf["item_id"].isin(batch_item_ids)
        ]
        orders_hist_batch = to_pandas_if_dask(orders_hist_batch_ddf)

        feature_tables_batch = build_feature_tables(
            tracker_pdf=tracker_hist_batch,
            orders_pdf=orders_hist_batch,
            item_meta=item_meta,
            ref_date=ref_date_val,
            cand_df=batch_cand,
        )

        val_feat_batch, _ = make_rerank_features(
            cand_df=batch_cand,
            feature_tables=feature_tables_batch,
            item_meta=item_meta,
            pop_table=pop_table_val,
            ref_date=ref_date_val,
        )

        scored_batch = rerank_candidates(
            df=val_feat_batch[["user_id", "item_id"] + feature_cols],
            feature_cols=feature_cols,
            ranker=ranker,
            topk=topk,
        )

        pred_map = (
            scored_batch.sort_values(["user_id", "rank"])
            .groupby("user_id")["item_id"]
            .apply(list)
            .to_dict()
        )

        for u in batch_user_ids:
            relevant = gt_map.get(u)
            if not relevant:
                continue
            s = ndcg_at_k(pred_map.get(u, []), relevant, k=topk)
            if s is not None:
                ndcg_scores.append(s)

        del (
            batch_cand,
            tracker_hist_batch_ddf,
            tracker_hist_batch,
            orders_hist_batch_ddf,
            orders_hist_batch,
            feature_tables_batch,
            val_feat_batch,
            scored_batch,
            pred_map,
        )
        gc.collect()

    return float(np.mean(ndcg_scores)) if ndcg_scores else np.nan

In [19]:
full_val_ndcg = evaluate_reranker_on_full_val_batched(
    union_val_candidates=union_val_candidates,
    labels_val=labels_val,
    tracker_df=tracker_df,
    orders_df=orders_df,
    item_meta=item_meta,
    pop_table_val=pop_table_val,
    ranker=ranker,
    feature_cols=feature_cols,
    ref_date_val=pd.to_datetime(TRAIN_END),
    batch_users=5000,
    topk=FINAL_TOPK,
)

print("full_val_ndcg =", full_val_ndcg)
print("candidate_recall_val =", cand_recall_val)
print("reduced_holdout_ndcg =", holdout_ndcg)

full val eval batches:   0%|          | 0/72 [00:00<?, ?it/s]

full_val_ndcg = 0.02564110010906468
candidate_recall_val = 0.04907423291119019
reduced_holdout_ndcg = 0.801065841404937


In [20]:
MODEL_DIR = Path("../data/interim/reranker_light")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ranker_path = MODEL_DIR / "catboost_reranker_light.cbm"
meta_path = MODEL_DIR / "catboost_reranker_light_meta.json"

ranker.save_model(str(ranker_path))

meta = {
    "feature_cols": feature_cols,
    "holdout_ndcg": float(holdout_ndcg),
    "final_iterations": int(final_iterations),
    "final_topk": int(FINAL_TOPK),
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False)

print("saved:")
print(ranker_path)
print(meta_path)

saved:
..\data\interim\reranker_light\catboost_reranker_light.cbm
..\data\interim\reranker_light\catboost_reranker_light_meta.json


## Final inference на full history

In [21]:
MODEL_DIR = Path("../data/interim/reranker_light")

ranker = CatBoostRanker()
ranker.load_model(str(MODEL_DIR / "catboost_reranker_light.cbm"))

with open(MODEL_DIR / "catboost_reranker_light_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

feature_cols = meta["feature_cols"]
print(meta)

{'feature_cols': ['in_als', 'in_pop', 'both_sources', 'source_count', 'als_score', 'als_rank_inv', 'pop_score', 'pop_rank_inv', 'global_pop_score', 'global_pop_rank_inv', 'ui_strength', 'ui_event_cnt', 'ui_strength_user_share', 'ui_days_since_last', 'user_total_strength', 'user_event_cnt', 'user_unique_items', 'user_days_since_last', 'user_is_cold', 'item_total_strength', 'item_event_cnt', 'item_unique_users', 'item_order_cnt', 'item_days_since_last', 'item_order_user_event_ratio', 'user_cat_strength', 'user_cat_share', 'same_cat_top3', 'log1p_als_score', 'log1p_pop_score', 'log1p_global_pop_score', 'log1p_ui_strength', 'log1p_user_total_strength', 'log1p_item_total_strength', 'log1p_ui_event_cnt', 'log1p_user_event_cnt', 'log1p_item_event_cnt', 'log1p_item_order_cnt', 'log1p_item_unique_users', 'log1p_user_unique_items', 'log1p_user_cat_strength'], 'holdout_ndcg': 0.801065841404937, 'final_iterations': 325, 'final_topk': 100}


In [22]:
als_full_artifacts = load_als_artifacts("full")
als_test_candidates = als_full_artifacts["candidates"].copy()
als_test_candidates = als_test_candidates[
    als_test_candidates["user_id"].isin(test_users)
].copy()

pop_table_full, pop_test_recs = load_pop_artifacts("full")
union_test_candidates = merge_candidate_sources(als_test_candidates, pop_test_recs)

Path("../data/interim/reranker_test_scored_batches").mkdir(parents=True, exist_ok=True)
union_test_candidates.to_parquet("../data/interim/reranker_union_test_candidates.parquet", index=False)

ref_date_full = pd.to_datetime(NEW_TRAIN_END)

BATCH_USERS = 5000
test_user_list = union_test_candidates["user_id"].drop_duplicates().astype(np.int64).tolist()

batch_paths = []

for batch_idx, start in enumerate(tqdm(range(0, len(test_user_list), BATCH_USERS), desc="test batches")):
    batch_users = test_user_list[start:start + BATCH_USERS]

    batch_cand = union_test_candidates[
        union_test_candidates["user_id"].isin(batch_users)
    ].copy()
    batch_items = batch_cand["item_id"].drop_duplicates().astype(np.int64).tolist()

    tracker_hist_batch_ddf = filter_tracker_ddf(
        tracker_df, NEW_TRAIN_START, NEW_TRAIN_END, keep_date=True
    )
    tracker_hist_batch_ddf = tracker_hist_batch_ddf[
        tracker_hist_batch_ddf["user_id"].isin(batch_users) |
        tracker_hist_batch_ddf["item_id"].isin(batch_items)
    ]
    tracker_hist_batch = to_pandas_if_dask(tracker_hist_batch_ddf)

    orders_hist_batch_ddf = filter_orders_ddf(
        orders_df, NEW_TRAIN_START, NEW_TRAIN_END, keep_created_date=True
    )
    orders_hist_batch_ddf = orders_hist_batch_ddf[
        orders_hist_batch_ddf["user_id"].isin(batch_users) |
        orders_hist_batch_ddf["item_id"].isin(batch_items)
    ]
    orders_hist_batch = to_pandas_if_dask(orders_hist_batch_ddf)

    feature_tables_batch = build_feature_tables(
        tracker_pdf=tracker_hist_batch,
        orders_pdf=orders_hist_batch,
        item_meta=item_meta,
        ref_date=ref_date_full,
        cand_df=batch_cand,
    )

    test_feat_batch, _ = make_rerank_features(
        cand_df=batch_cand,
        feature_tables=feature_tables_batch,
        item_meta=item_meta,
        pop_table=pop_table_full,
        ref_date=ref_date_full,
    )

    test_scored_batch = rerank_candidates(
        df=test_feat_batch[["user_id", "item_id"] + feature_cols],
        feature_cols=feature_cols,
        ranker=ranker,
        topk=FINAL_TOPK,
    )

    batch_path = f"../data/interim/reranker_test_scored_batches/batch_{batch_idx:04d}.parquet"
    test_scored_batch.to_parquet(batch_path, index=False)
    batch_paths.append(batch_path)

    del (
        batch_cand,
        batch_items,
        tracker_hist_batch_ddf,
        tracker_hist_batch,
        orders_hist_batch_ddf,
        orders_hist_batch,
        feature_tables_batch,
        test_feat_batch,
        test_scored_batch,
    )
    gc.collect()

print(f"ALS test candidates rows: {len(als_test_candidates):,}")
print(f"POP test candidates rows: {len(pop_test_recs):,}")
print(f"UNION test candidates rows: {len(union_test_candidates):,}")
print(f"Saved test batches: {len(batch_paths):,}")
print(batch_paths[:3])

test batches:   0%|          | 0/70 [00:00<?, ?it/s]

ALS test candidates rows: 69,134,600
POP test candidates rows: 34,675,900
UNION test candidates rows: 98,102,950
Saved test batches: 70
['../data/interim/reranker_test_scored_batches/batch_0000.parquet', '../data/interim/reranker_test_scored_batches/batch_0001.parquet', '../data/interim/reranker_test_scored_batches/batch_0002.parquet']


In [23]:
test_scored = pd.concat([pd.read_parquet(p) for p in batch_paths], ignore_index=True)
print(f"Scored test rows: {len(test_scored):,}")
test_scored.head()

Scored test rows: 34,675,900


,user_id,item_id,in_als,in_pop,both_sources,source_count,als_score,als_rank_inv,pop_score,pop_rank_inv,...,log1p_item_total_strength,log1p_ui_event_cnt,log1p_user_event_cnt,log1p_item_event_cnt,log1p_item_order_cnt,log1p_item_unique_users,log1p_user_unique_items,log1p_user_cat_strength,rerank_score,rank
0,1,65503348,1.0,0.0,0.0,1.0,0.219363,0.005025,0.0,0.0,...,8.146643,0.000000,7.932003,6.840547,3.044523,5.899898,7.379008,2.442347,9.120700,1
1,1,290412994,1.0,0.0,0.0,1.0,0.219149,0.005000,0.0,0.0,...,7.801373,0.693147,7.932003,7.163947,2.564949,6.240276,7.379008,0.926779,9.108166,2
2,1,48385895,1.0,0.0,0.0,1.0,0.225473,0.005747,0.0,0.0,...,8.275846,1.386294,7.932003,7.403670,2.197225,6.562444,7.379008,1.556298,9.084542,3
3,1,309232136,1.0,0.0,0.0,1.0,0.231696,0.006410,0.0,0.0,...,7.752140,0.000000,7.932003,6.436151,1.791759,5.560682,7.379008,0.259994,9.084183,4
4,1,249956118,1.0,0.0,0.0,1.0,0.223163,0.005525,0.0,0.0,...,8.079015,0.693147,7.932003,7.296413,2.995732,6.400258,7.379008,0.259994,9.080590,5


In [24]:
user2items_test = maybe_build_test_gt(orders_ddf_test)

if user2items_test is not None:
    labels_test = pd.DataFrame(
        [(u, i) for u, items in user2items_test.items() for i in items],
        columns=["user_id", "item_id"],
    ).drop_duplicates()

    test_ndcg = evaluate_ranked_df(
        scored_df=test_scored,
        labels_df=labels_test,
        k=FINAL_TOPK,
    )
    print(f"Test NDCG@{FINAL_TOPK}: {test_ndcg:.5f}")
else:
    test_ndcg = None
    print("Test metric skipped: в load_orders(train=False) нет доступного ground truth.")

Test NDCG@100: 0.02908


## Сохранение артефактов

In [25]:
Path("../data/interim").mkdir(parents=True, exist_ok=True)
Path("../models").mkdir(parents=True, exist_ok=True)

# candidates
als_val_candidates.to_parquet("../data/interim/als_val_candidates_used_by_reranker.parquet", index=False)
als_test_candidates.to_parquet("../data/interim/als_test_candidates_used_by_reranker.parquet", index=False)

union_val_candidates.to_parquet("../data/interim/union_val_candidates.parquet", index=False)
union_test_candidates.to_parquet("../data/interim/union_test_candidates.parquet", index=False)

# train data
train_basis_val.to_parquet("../data/interim/rerank_train_basis_light.parquet", index=False)
train_feat_val.to_parquet("../data/interim/rerank_train_feat_light.parquet", index=False)

# scored results
train_scored_val.to_parquet("../data/interim/rerank_val_scored.parquet", index=False)
test_scored.to_parquet("../data/interim/rerank_test_scored.parquet", index=False)

# model
ranker.save_model("../models/rerank_model_light.cbm")

with open("../models/rerank_feature_cols_light.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

metrics = {
    "holdout_ndcg": None if pd.isna(holdout_ndcg) else float(holdout_ndcg),
    "train_ndcg": None if pd.isna(train_ndcg) else float(train_ndcg),
    "test_ndcg": None if (test_ndcg is None or pd.isna(test_ndcg)) else float(test_ndcg),
    "candidate_recall_val": None if pd.isna(cand_recall_val) else float(cand_recall_val),
    "n_train_basis_rows": int(len(train_basis_val)),
    "n_train_feat_rows": int(len(train_feat_val)),
    "n_test_scored_rows": int(len(test_scored)),
    "n_features": int(len(feature_cols)),
    "final_iterations": int(final_iterations),
}

with open("../models/rerank_metrics_light.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Сохранено:")
print(" - ../data/interim/als_val_candidates_used_by_reranker.parquet")
print(" - ../data/interim/als_test_candidates_used_by_reranker.parquet")
print(" - ../data/interim/union_val_candidates.parquet")
print(" - ../data/interim/union_test_candidates.parquet")
print(" - ../data/interim/rerank_train_basis_light.parquet")
print(" - ../data/interim/rerank_train_feat_light.parquet")
print(" - ../data/interim/rerank_val_scored.parquet")
print(" - ../data/interim/rerank_test_scored.parquet")
print(" - ../models/rerank_model_light.cbm")
print(" - ../models/rerank_feature_cols_light.json")
print(" - ../models/rerank_metrics_light.json")
print()
print(metrics)

Сохранено:
 - ../data/interim/als_val_candidates_used_by_reranker.parquet
 - ../data/interim/als_test_candidates_used_by_reranker.parquet
 - ../data/interim/union_val_candidates.parquet
 - ../data/interim/union_test_candidates.parquet
 - ../data/interim/rerank_train_basis_light.parquet
 - ../data/interim/rerank_train_feat_light.parquet
 - ../data/interim/rerank_val_scored.parquet
 - ../data/interim/rerank_test_scored.parquet
 - ../models/rerank_model_light.cbm
 - ../models/rerank_feature_cols_light.json
 - ../models/rerank_metrics_light.json

{'holdout_ndcg': 0.801065841404937, 'train_ndcg': 0.09953298777590644, 'test_ndcg': 0.02907847516181152, 'candidate_recall_val': 0.04907423291119019, 'n_train_basis_rows': 21650820, 'n_train_feat_rows': 21650820, 'n_test_scored_rows': 34675900, 'n_features': 41, 'final_iterations': 325}
